In [1]:
import os
import json
import random
import numpy as np
import torch

from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [7]:
!git clone https://github.com/lxcentry/cu_konk_stip_ml_red_task.git

Cloning into 'cu_konk_stip_ml_red_task'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 31 (delta 3), reused 29 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 5.23 MiB | 11.51 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [8]:
!pip install -q -U transformers accelerate peft trl bitsandbytes huggingface_hub

In [10]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen3-4B-Instruct-2507",
    local_dir="./qwen3_4b",
    local_dir_use_symlinks=False,
)

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

'/content/qwen3_4b'

In [11]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

MODEL_NAME = "./qwen3_4b"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [12]:
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
)

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 132,120,576 || all params: 4,154,588,672 || trainable%: 3.1801


In [14]:
import json
from datasets import Dataset

examples = []

with open("cu_konk_stip_ml_red_task/ml-olympiad-red-task/data/kid_adult.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)

        messages = [
            {"role": "user", "content": row["prompt"]},
            {"role": "assistant", "content": row["kid"]},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        examples.append({"text": text})

train_dataset = Dataset.from_list(examples)

print(train_dataset[0])

{'text': '<|im_start|>user\nКак работает камера видеонаблюдения? Ответ: Она ловит свет через стеклышко, превращает его в электрические сигналы и сохраняет их как видео в памяти.<|im_end|>\n<|im_start|>assistant\nКамера работает как волшебный глаз. Она улавливает свет через свое стеклышко, переводит его в электрические сигналы и сохраняет как видео в памяти.<|im_end|>\n'}


In [ ]:
from trl import SFTTrainer
import inspect

print(inspect.signature(SFTTrainer))